# 从模型到芯片

1. 加载已经训练好的 PyTorch MobileNetV2（有猫 / 无猫）模型；
2. 导出为 ONNX 模型；
3. 将 ONNX 模型转换为 TFLite int8 量化模型；
4. 在 Colab 上用 TFLite Interpreter 验证转换前后模型输出是否一致。


## Step 0：挂载 Drive & 安装/导入依赖

目标：
- 挂载 Google Drive，用于读取训练好的模型和保存中间文件；
- 安装 onnx / onnx2tf / tensorflow 等必要库；
- 导入 PyTorch / torchvision / ONNX / TensorFlow 等库。


In [ ]:
!pip install onnx onnx2tf tensorflow

In [ ]:
# Step 0: 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 1: 导入库
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2
import torchvision.transforms as T
import onnx
import onnx2tf
import tensorflow as tf
import numpy as np
from PIL import Image
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Step 1：加载训练好的 PyTorch MobileNetV2 模型

目标：
- 从 `.pth` 文件中加载你之前训练好的 MobileNetV2（2 类：有猫 / 无猫）；
- 保证模型结构和训练时一致；
- 切换到 `eval()` 模式；
- 准备一个 `preprocess_image(path)` 函数，用来对测试图片做预处理。


In [ ]:
# Step 2: 定义模型结构并加载训练好的权重
model = mobilenet_v2(weights=None)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)  # 2 类：猫/非猫

pth_path = "/content/drive/MyDrive/catbox/models/mobilenet_v2_final.pth"
model.load_state_dict(torch.load(pth_path, map_location=device))
model.eval()
model.to(device)



## Step 2：导出 ONNX 模型

目标：
- 使用 `torch.onnx.export` 将当前 PyTorch 模型导出为 ONNX 文件；
- 指定 dummy_input 的形状（例如 `[1, 3, image_size, image_size]`）；
- 用 `onnx.load` 和 `onnx.checker.check_model` 验证导出的 ONNX 模型结构正常。


In [ ]:
# Step 3: 导出 ONNX 模型
onnx_path = "/content/drive/MyDrive/catbox/models/mobilenet_v2.onnx"
dummy_input = torch.randn(1, 3, 224, 224, device=device)
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    opset_version=12
)
print("ONNX exported to:", onnx_path)
# Step 4: 检查 ONNX 模型
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model is valid!")



## Step 3：ONNX → TFLite (int8 量化)

目标：
- 使用 onnx2tf 等工具将 ONNX 模型转换为 TensorFlow 格式；
- 使用 TFLiteConverter 将 TensorFlow 模型转换为 int8 量化的 `.tflite` 文件；
- 注意：int8 全量化需要一个代表性数据集（Representative Dataset）。


In [ ]:
# Step 3：ONNX → TF → TFLite int8

# 提示：可以使用 vibe coding 询问 AI：
# “如何用 onnx2tf 将 onnx 模型转换为 SavedModel？”
# “如何用 TFLiteConverter 从 SavedModel 转成 int8 模型？”


# TODO: 指定 TensorFlow SavedModel 输出目录

# TODO: 使用 onnx2tf 将 onnx_path 转换为 tf_model_dir


# TODO: 使用 TFLiteConverter.from_saved_model 加载 tf_model_dir

# TODO: 设置优化选项为默认

# TODO: 定义代表性数据集函数 representative_dataset()
# 用你自己的部分训练图片做若干 batch，返回 float32 输入


# TODO: 配置为全 int8 量化


# TODO: 执行转换


# TODO: 保存为 .tflite 文件

# Step 5: ONNX → TensorFlow SavedModel
tf_model_dir = "/content/drive/MyDrive/catbox/models/tf_saved_model"
onnx2tf.convert(
    onnx_path,
    tf_model_dir
)
print("Converted to TensorFlow SavedModel at:", tf_model_dir)


# Step 6: TensorFlow → TFLite (int8 量化)
tflite_model_path = "/content/drive/MyDrive/catbox/models/mobilenet_v2_int8.tflite"

# 代表性数据集函数
def representative_dataset():
    for _ in range(10):  # 用 10 张图片做量化
        data = np.random.rand(1, 224, 224, 3).astype(np.float32)
        yield [data]

converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

# 保存 TFLite 模型
with open(tflite_model_path, "wb") as f:
    f.write(tflite_model)
print("TFLite int8 model saved to:", tflite_model_path)


## Step 4：在 Colab 上验证 TFLite 模型推理结果

目标：
- 使用 TFLite Interpreter 加载 int8 模型；
- 对同一张 test 图片：分别用 PyTorch 模型和 TFLite 模型做推理；
- 对比两者的预测类别是否一致，观察量化前后差异。


In [ ]:
# Step 4：加载 TFLite 模型并做一次推理

# TODO: 创建 TFLite Interpreter

# TODO: 获取 input/output 信息


# 对 int8 模型来说，通常需要做量化：
# 输入: float32 → 按 scale & zero_point 转成 int8
# 输出: int8 → 可按 scale & zero_point 反量化，或直接比较 argmax

# TODO: 用与 PyTorch 那边相同的 preprocess 得到 float32 的 [1,C,H,W] 张量
# 注意 TFLite 通常的输入是 [1,H,W,C] NHWC 排布，所以需要转置

# TODO: 根据 input_details[0]['quantization'] 做量化


# TODO: 设置输入张量并调用 interpreter.invoke()


# TODO: 如果需要，可以根据 output_details[0]['quantization'] 反量化，
# 或直接比较 argmax


# TODO: 对比 PyTorch 模型在同一张图上的预测类别是否一致


## Step 5：小结 & 下一步

本 Notebook 中，你已经完成：

- 从 PyTorch MobileNetV2 模型导出 ONNX；
- 使用 onnx2tf + TFLiteConverter 得到一个 int8 量化的 TFLite 模型；
- 在 Colab 上使用 TFLite Interpreter 对测试图片做推理，并和 PyTorch 输出做对比。

下一步可以考虑：

- 在本地用 `xxd -i mobilenetv2_catbox_int8.tflite > model_data.cc` 将模型转换为 C 数组；
- 在 Arduino / ESP32-S3 上使用 TFLite Micro 加载 `g_model[]`，做一次简单的推理（固定输入 → 输出结果打印到串口）。


# 从TFLite到XIAO ML Kit

## 一、XIAO ML Kit 上整合 TFLite 模型的步骤

### 1. 准备工作：转换 TFLite 模型为 C 数组

首先，确保你已经：

* **在 Colab 上完成了模型转换**：
  从 PyTorch → ONNX → TFLite，并完成了 int8 量化；
* **生成了 `model_data.cc` 文件**：
  用 `xxd -i` 或其他工具把 `.tflite` 文件转为 C 数组。
  例如：

  ```bash
  xxd -i mobilenetv2_catbox_int8.tflite > model_data.cc
  ```

在 Arduino 环境中，我们将这个 `.cc` 文件作为模型数据的一部分，加载到 `g_model[]` 数组中。


## 2. 配置 XIAO 的开发环境

在 XIAO 上运行模型，我们需要：

1. **安装 TensorFlow Lite Micro**：

   * 你可以通过 **Arduino IDE** 或 **ESP-IDF** 来编译 TFLite Micro。
   * 推荐在 **Arduino IDE** 中使用 **TensorFlowLite_ESP32** 库（Seeed 官方有支持）。
   * `TensorFlowLite_ESP32` 是专门为 ESP32 硬件优化的轻量级 TFLite 实现。

2. **准备 Arduino 代码结构**：

   * `model_data.cc`：包含了 TFLite 模型；
   * `TensorArena`：TFLite 微解释器需要的一块内存区域；
   * `TensorFlow Lite MicroInterpreter`：用于加载和推理。


## 二、整合步骤

### 1. 创建 Arduino 项目结构

在 Arduino IDE 中创建项目：

1. **`sketch.ino`**：这是主要的代码文件，包含了：

   * 初始化 WiFi 或串口；
   * 加载 TFLite 模型；
   * 执行推理。

2. **`model_data.cc`**：
   通过 `xxd -i` 生成的模型数据，存储为一个 `unsigned char g_model[]` 数组。

3. **`TensorFlowLite_ESP32.h`**：
   引入 TensorFlow Lite Micro 库，提供推理功能。



### 2. Arduino 代码框架

```cpp
#include <TensorFlowLite_ESP32.h>  // 引入 TFLite Micro 库
#include "model_data.cc"           // 引入模型数据

// 1. 配置 TFLite 相关对象
tflite::MicroErrorReporter micro_error_reporter;
const tflite::Model* model = nullptr;
tflite::MicroInterpreter* interpreter = nullptr;
TfLiteTensor* input = nullptr;
TfLiteTensor* output = nullptr;

// 2. 配置 TensorArena（分配内存）
constexpr int kTensorArenaSize = 200 * 1024;  // 给定一个 200KB 的 TensorArena
static uint8_t tensor_arena[kTensorArenaSize];

// 3. 连接 WiFi 或初始化串口
void setup() {
  Serial.begin(115200);
  delay(1000);

  // 4. 加载 TFLite 模型
  model = tflite::GetModel(g_model);
  if (model->version() != TFLITE_SCHEMA_VERSION) {
    Serial.println("Model schema version mismatch!");
    while (1);
  }

  // 5. 创建 ops resolver
  static tflite::AllOpsResolver resolver;

  // 6. 创建解释器
  static tflite::MicroInterpreter static_interpreter(
      model, resolver, tensor_arena, kTensorArenaSize, &micro_error_reporter);
  interpreter = &static_interpreter;

  // 7. 分配张量
  TfLiteStatus allocate_status = interpreter->AllocateTensors();
  if (allocate_status != kTfLiteOk) {
    Serial.println("AllocateTensors() failed");
    while (1);
  }

  // 8. 获取输入输出张量指针
  input = interpreter->input(0);
  output = interpreter->output(0);

  Serial.println("TFLite model initialized.");
}

void loop() {
  // 9. 填充输入张量（这里我们暂时用固定值作为输入测试）
  for (int i = 0; i < input->bytes; i++) {
    input->data.uint8[i] = 127;  // 填充数据，可以替换为真实图像数据
  }

  // 10. 执行推理
  TfLiteStatus invoke_status = interpreter->Invoke();
  if (invoke_status != kTfLiteOk) {
    Serial.println("Invoke failed!");
    return;
  }

  // 11. 打印输出结果
  Serial.print("Output[0] = ");
  Serial.print(output->data.f[0]);  // 假设是第一个输出类别
  Serial.print(", Output[1] = ");
  Serial.println(output->data.f[1]);  // 假设是第二个输出类别

  delay(2000);  // 每隔 2 秒执行一次推理
}
```

### 3. 代码解析

1. **模型加载：**
   `model = tflite::GetModel(g_model);`
   这里的 `g_model` 就是你通过 `xxd -i` 转换的 `model_data.cc` 数组，包含了 TFLite 模型的所有数据。

2. **内存分配：**
   `interpreter->AllocateTensors();`
   为 TFLite 模型分配内存，这个操作在解释器里进行，分配张量内存和推理所需的计算资源。

3. **执行推理：**
   `interpreter->Invoke();`
   调用 `Invoke()`，TFLite 会在输入张量上运行推理，输出结果保存在 `output` 中。

4. **输出结果：**
   `output->data.f[0]` 和 `output->data.f[1]`：
   假设模型有两个输出节点（“有猫”和“无猫”概率），我们从 `output` 中提取推理结果。
